# Analytica DeepAgents notebook smoke test

This notebook tests the current project agent through the project entrypoints, not through private DeepAgents internals. It is safe to execute without API keys: real LLM execution is gated behind an environment flag.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('project root:', PROJECT_ROOT)
assert (PROJECT_ROOT / 'source').exists(), 'source/ package not found'


In [ ]:
from source.config import DEFAULT_DATA_PATH
from source.dataframe import read_csv_dataset, dataframe_profile

# The demo CSV contains escaped delimiters, so pass escapechar explicitly.
# Restart the Jupyter kernel after code changes if imports look stale.
df = read_csv_dataset(DEFAULT_DATA_PATH, escapechar=chr(92))
profile = dataframe_profile(df)
print('dataset path:', DEFAULT_DATA_PATH)
print('shape:', df.shape)
print('columns:', list(df.columns))
assert not df.empty
df.head()


In [ ]:
skills_dir = PROJECT_ROOT / 'source' / 'skills'
skill_files = sorted(skills_dir.glob('*/SKILL.md'))
skill_names = []
for skill_file in skill_files:
    text = skill_file.read_text(encoding='utf-8')
    assert text.startswith('---'), f'missing frontmatter: {skill_file}'
    frontmatter = text.split('---', 2)[1]
    assert 'name:' in frontmatter, f'missing name: {skill_file}'
    assert 'description:' in frontmatter, f'missing description: {skill_file}'
    for line in frontmatter.splitlines():
        if line.strip().startswith('name:'):
            skill_names.append(line.split(':', 1)[1].strip())

print('expected skill source path:', '/source/skills/')
print('expected memory file:', '/memories/AGENTS.md')
print('skills:', skill_names)
assert (PROJECT_ROOT / 'source' / 'skills').is_dir()
assert (PROJECT_ROOT / '.analytica' / 'memory').exists() or True
assert len(skill_names) >= 3


In [ ]:
import importlib
import inspect
import source.config as config_module

# Reload project modules so Jupyter does not use stale objects after source edits.
config_module = importlib.reload(config_module)
assert hasattr(config_module, 'DEEPAGENTS_MEMORY_DIR')
assert hasattr(config_module, 'DEEPAGENTS_MEMORY_FILE')

import source.agent as agent_module
agent_module = importlib.reload(agent_module)
module_source = inspect.getsource(agent_module)
build_signature = inspect.signature(agent_module.build_deep_agent)

assert hasattr(agent_module, 'build_deep_agent')
assert hasattr(agent_module, 'run_agent')
assert 'create_deep_agent' in module_source
assert 'build_analytics_tools()' in module_source
assert 'skills=' in module_source and '/source/skills/' in module_source
assert 'memory=' in module_source and '/memories/AGENTS.md' in module_source
assert 'context_schema=AnalyticaContext' in module_source
assert 'skill_descriptions_text' not in module_source
print('config memory dir:', config_module.DEEPAGENTS_MEMORY_DIR)
print('DeepAgents construction path looks official')
print('build_deep_agent signature:', build_signature)


In [ ]:
import importlib
import source.agent as agent_module
agent_module = importlib.reload(agent_module)

from source.runtime_context import AnalyticaContext
from source.tools.analytics_tools import build_analytics_tools

run_context = {
    'query': 'Notebook smoke test',
    'df': df,
    'engine': 'pandas',
    'schema': '',
    'tool_timeline': [],
}
runtime_context = agent_module._agent_runtime_context(run_context, 'notebook-smoke')
tools = build_analytics_tools(run_context)
tools_by_name = {tool.name: tool for tool in tools}

print('context:', runtime_context)
print('tools:', list(tools_by_name))
assert isinstance(runtime_context, AnalyticaContext)
assert 'inspect_dataset_schema' in tools_by_name
assert 'run_python_analysis' in tools_by_name
assert 'query_dataframe_sql' in tools_by_name


In [ ]:
from textwrap import dedent

tool_names = list(tools_by_name) if 'tools_by_name' in globals() else [tool.name for tool in build_analytics_tools()]
schema_text = f"""
ANALYTICA AGENT SCHEMA
======================

Official DeepAgents construction
--------------------------------
create_deep_agent(
    model=make_llm("deep_agent"),
    tools=build_analytics_tools(),
    skills=["/source/skills/"],
    memory=["/memories/AGENTS.md"],
    backend=CompositeBackend(...),
    context_schema=AnalyticaContext,
    checkpointer=get_checkpointer(),
)

Backend routes
--------------
default        -> StateBackend() for thread-scoped scratch state
/artifacts/    -> FilesystemBackend(root_dir=artifacts, virtual_mode=True)
/memories/     -> FilesystemBackend(root_dir=.analytica/memory, virtual_mode=True)
/source/skills/ -> FilesystemBackend(root_dir=source/skills, virtual_mode=True)

Runtime context
---------------
AnalyticaContext(
    user_id, thread_id, session_id, run_id, artifact_dir, memory_file, run_state
)

Skills
------
{chr(10).join(f'- {name}' for name in skill_names)}

Tools
-----
{chr(10).join(f'- {name}' for name in tool_names)}

User-facing run flow
--------------------
1. User question
2. DeepAgents selects relevant skills and tools
3. inspect_dataset_schema reads dataframe schema/profile
4. run_python_analysis / query_dataframe_sql / plot_bar produce evidence
5. artifacts and tool_timeline are recorded in run_state
6. run_agent returns final_answer + structured_report + artifacts + trace metadata

Notebook test coverage
----------------------
- skill folder/frontmatter check
- official create_deep_agent setup check
- typed runtime context check
- Python tool smoke test
- SQL check + query smoke test
- deterministic run_agent / run_agent_stream smoke test
- optional real LLM run behind ANALYTICA_RUN_NOTEBOOK_LLM=1
"""
print(dedent(schema_text).strip())


## Agent Flow Diagram

```mermaid
flowchart LR
    Q[User question] --> A[DeepAgents agent]
    A --> S[Skills: /source/skills/]
    A --> C[AnalyticaContext]
    A --> T[Analytics tools]
    T --> D[DataFrame / SQL / Python analysis]
    T --> R[run_state: artifacts + tool_timeline]
    R --> O[final_answer + structured_report]
    O --> P[Investigation / Report product layer]
```


In [ ]:
schema_result = tools_by_name['inspect_dataset_schema'].func(None)
print(schema_result['engine'])
print(schema_result['schema'][:500])
assert 'Sales' in schema_result['schema']

python_result = tools_by_name['run_python_analysis'].func(
    "result = df.loc[df['City'].eq('Los Angeles'), 'Sales'].mean()",
    None,
)
print('python result:', python_result)
assert python_result['exec_error'] in ('', None)
assert python_result['result_preview']

sql_query = "SELECT City, AVG(Sales) AS mean_sales FROM data GROUP BY City ORDER BY mean_sales DESC LIMIT 5"
sql_check = tools_by_name['check_dataframe_sql'].func(sql_query, None)
print('sql check:', sql_check)
assert sql_check['valid'] == 'true'
sql_result = tools_by_name['query_dataframe_sql'].func(sql_query, None)
print('sql result:', sql_result['result_preview'])
assert sql_result['exec_error'] in ('', None)
assert 'mean_sales' in sql_result['result_preview']

print('tool timeline:', run_context['tool_timeline'])
assert len(run_context['tool_timeline']) >= 4


In [ ]:
from source.agent import run_agent, run_agent_stream

question = 'Опиши структуру данных и возможные направления анализа'
output = run_agent(df, question, engine='pandas', thread_id='notebook-smoke-agent')
print(output['final_answer'][:1000])
print('selected skills:', output['selected_skills'])
print('tool timeline:', output['tool_timeline'])
assert output['final_answer']
assert output['structured_report']['summary']
assert output['tool_timeline']

events = list(run_agent_stream(df, question, engine='pandas', thread_id='notebook-smoke-stream'))
print([(event['event'], event['stage']) for event in events])
assert events[-1]['event'] == 'final'
assert events[-1]['output']['final_answer']


In [ ]:
# Optional: set ANALYTICA_RUN_NOTEBOOK_LLM=1 and configure an LLM provider/API key
# to run a real DeepAgents analysis from this notebook. This cell is skipped by default
# so CI/local smoke execution does not depend on external services.
if os.getenv('ANALYTICA_RUN_NOTEBOOK_LLM') == '1':
    real_question = 'What is the mean of Sales for orders in Los Angeles?'
    real_output = run_agent(df, real_question, engine='pandas', thread_id='notebook-real-agent')
    print(real_output['final_answer'])
    print(real_output.get('result_preview', ''))
    assert real_output['final_answer']
else:
    print('Skipped real LLM run. Set ANALYTICA_RUN_NOTEBOOK_LLM=1 to enable it.')
